# 💊 SHAP

SHAP (SHapley Additive exPlanations)는 머신러닝 모델의 출력을 설명하는 데 사용되는 방법입니다.  
SHAP는 각 입력 특성이 출력에 미치는 영향을 보여줌으로써 입력이 모델의 출력에 어떻게 영향을 미치는지 설명하려고 합니다.  
SHAP 값을 읽을 때 각 입력 특성에 대해 데이터셋의 평균 기본값과 비교하여 우리가 얻은 답변으로 출력을 얼마나 긍정적으로 또는 부정적으로 밀어냈는지 볼 수 있습니다.

자세한 내용은 여기에서 확인할 수 있습니다: https://trustyai-explainability.github.io/trustyai-site/main/local-explainers.html

In [ ]:
!pip -q install "onnx" "onnxruntime" "numpy==1.26.4"

In [ ]:
import pandas as pd
import onnxruntime as rt
import numpy as np
import pickle

In [ ]:
import warnings

# UserWarning 무시
warnings.filterwarnings("ignore", category=UserWarning)

몇 가지 아티팩트를 로드하는 것부터 시작합시다.  
우리는 다음이 필요합니다:
- ONNX 모델
- 우리의 사전 및 사후 처리 아티팩트
    - scaler.pkl
    - label_encoder.pkl
- 일부 데이터
    - 학습 입력, 데이터셋의 평균 입력을 얻는 데 사용됩니다
    - 테스트 데이터, 분석하려는 포인트를 얻는 데 사용됩니다

In [ ]:
onnx_session = rt.InferenceSession("convert-keras-to-onnx/onnx_model.onnx", providers=rt.get_available_providers())
onnx_input_name = onnx_session.get_inputs()[0].name
onnx_output_name = onnx_session.get_outputs()[0].name


with open('preprocess-data/scaler.pkl', 'rb') as handle:
    scaler = pickle.load(handle)

with open('preprocess-data/label_encoder.pkl', 'rb') as handle:
    label_encoder = pickle.load(handle)

with open('preprocess-data/train_data.pkl', 'rb') as handle:
    train_data = pickle.load(handle)

with open('preprocess-data/test_data.pkl', 'rb') as handle:
    test_data = pickle.load(handle)

X_test = test_data[0]
X_train_scaled = pd.DataFrame(data=train_data[0], columns=X_test.columns) # X_train은 학습 파이프라인의 전처리 단계에서 확장됩니다

우리는 테스트 데이터의 첫 번째 데이터 포인트(노래)를 테스트하려는 데이터로 임의로 선택합니다.  
실제로는 최악의 예측을 하는 데이터 포인트를 선택하거나 예상치 못한 답변을 제공한 데이터 포인트를 선택할 수도 있습니다.  
우리는 또한 정규화된(전처리를 거친) 우리의 데이터 포인트가 어떻게 보이는지 봅니다. 이는 모델로 들어갈 때의 모습입니다

In [ ]:
point_to_explain = X_test.iloc[0:1]
point_to_explain

In [ ]:
def normalize_dataframe(df):
    normalized_data = scaler.transform(df)
    return pd.DataFrame(normalized_data, columns=df.columns)

In [ ]:
normalize_dataframe(point_to_explain)

사후 처리 아티팩트 label_encoder에서 모든 국가 코드를 가져옵니다.  
우리는 어떤 출력이 어떤 국가를 나타내는지 알기 위해 이들을 사용할 것입니다.

In [ ]:
output_names = label_encoder.classes_
output_names

TrustyAI SHAP 설명기는 모델이 pandas dataframe을 입력으로 가지고 numpy 또는 pandas 출력을 가지도록 요구하므로, 우리의 모델을 pred() 함수로 래핑하여 입력과 출력이 제대로 변환되도록 합니다. 

In [ ]:
def pred(x):
    x_dict = {name: x[[name]].to_numpy().astype(np.float32) for name in x.columns}
    pred = onnx_session.run([onnx_output_name], x_dict)[0]
    return pd.DataFrame(pred, columns=output_names)

In [ ]:
from trustyai.model import Model
trustyai_model = Model(pred, dataframe_input=True, output_names=output_names)

우리의 TrustyAI 모델을 사용하여 우리가 설명하려는 데이터 포인트의 출력을 예측해 봅시다.

In [ ]:
prediction = trustyai_model(normalize_dataframe(point_to_explain))
prediction

모든 것이 설정되었으므로 SHAP 설명기를 생성하고 우리의 데이터 포인트를 분석할 수 있습니다!  
또한 학습 데이터셋의 100개 데이터 포인트를 SHAPExplainer에 추가한다는 점을 주목할 수 있습니다. 이는 우리 데이터셋의 평균 기본값을 계산하는 데 사용됩니다. 이를 통해 우리의 흥미로운 데이터 포인트가 "표준" 값이 어떻게 예측되는지와 비교하여 얼마나 기여하는지 볼 수 있습니다.

In [ ]:
from trustyai.explainers import SHAPExplainer
explainer = SHAPExplainer(background=X_train_scaled[:100])

In [ ]:
explanations = explainer.explain(inputs=normalize_dataframe(point_to_explain),
                                 outputs=prediction,
                                 model=trustyai_model)

우리의 SHAP 설명기가 준비되어 있으므로 결과를 살펴볼 수 있습니다.

입력값이 영향을 미쳤는지 알고 싶은 특정 출력 국가를 선택해 봅시다.  
CH는 이 입력에 대해 인기 있는 국가로 나와야 하는 국가이므로 입력의 그 출력에 대한 영향을 보는 것이 특히 흥미로워집니다.  
어쨌든, 다른 몇 국가와 함께 시도해 보고 어떤 일이 일어나는지 보세요.  

In [ ]:
COUNTRY_OF_INTEREST = "CH"

먼저 값 테이블을 가져올 것입니다.  
여기서 우리는 **평균 배경값** 을 볼 수 있습니다 - 이는 우리가 전에 이야기한 평균 기본값입니다.  
또한 우리는 우리의 **값** 을 볼 수 있으며, 이는 우리가 설명기에 보낸 정규화된 데이터 포인트입니다. 빨간색 값은 평균값보다 낮고 녹색 값은 더 높습니다.  
마지막으로, 우리는 **SHAP 값** 을 가지고 있습니다. 이들은 그 입력 특성이 출력에 미친 영향을 나타냅니다. 빨간색은 예측에 대한 부정적인 기여를 나타내고 녹색은 긍정적인 기여를 나타냅니다. 값이 클수록 기여가 더 큽니다.

In [ ]:
explanations.as_html()[COUNTRY_OF_INTEREST]

우리는 또한 촛대 플롯으로 시각화할 수 있어서 다양한 입력 특성이 출력값을 어떻게 구축하는지 볼 수 있습니다.

In [ ]:
from trustyai.visualizations.shap import SHAPViz
SHAPViz()._matplotlib_plot(explanations=explanations, output_name=COUNTRY_OF_INTEREST)

### 퀴즈 시간 🤓

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../.dontlookhere/'))
from quiz4 import *

In [ ]:
quiz_shap()